In [1]:
!apt-get install graphviz
!pip install graphviz

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
graphviz is already the newest version (2.42.2-9ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.


In [2]:
import os
import time
import graphviz
from google.colab import drive

drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/Trabalho_Arvores/Treap'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Pasta de destino pronta em: {OUTPUT_DIR}")

Mounted at /content/drive
Pasta de destino pronta em: /content/drive/MyDrive/Trabalho_Arvores/Treap


In [3]:
import random

class TreapNode:
    def __init__(self, key):
        self.key = key
        self.priority = random.randint(1, 100000)
        self.left = None
        self.right = None

class TreapInstrumentada:
    def __init__(self):
        self.root = None
        self.comparisons_count = 0
        self.rotations_count = 0
        self.nodes_count = 0

    def reset_metrics(self):
        self.comparisons_count = 0
        self.rotations_count = 0

    def _rotate_right(self, y):
        self.rotations_count += 1
        x = y.left
        T2 = x.right
        x.right = y
        y.left = T2
        return x

    def _rotate_left(self, x):
        self.rotations_count += 1
        y = x.right
        T2 = y.left
        y.left = x
        x.right = T2
        return y

    def insert(self, key):
        def _insert(node, key):
            if not node:
                self.nodes_count += 1
                return TreapNode(key)

            self.comparisons_count += 1
            if key < node.key:
                node.left = _insert(node.left, key)
                if node.left.priority > node.priority:
                    node = self._rotate_right(node)
            elif key > node.key:
                node.right = _insert(node.right, key)
                if node.right.priority > node.priority:
                    node = self._rotate_left(node)

            return node

        self.root = _insert(self.root, key)

    def search(self, key) -> bool:
        curr = self.root
        while curr:
            self.comparisons_count += 1
            if curr.key == key:
                return True
            elif key < curr.key:
                curr = curr.left
            else:
                curr = curr.right
        return False

    def delete(self, key):
        def _delete(node, key):
            if not node:
                return node

            self.comparisons_count += 1
            if key < node.key:
                node.left = _delete(node.left, key)
            elif key > node.key:
                node.right = _delete(node.right, key)
            else:
                self.nodes_count -= 1
                if not node.left:
                    return node.right
                elif not node.right:
                    return node.left
                elif node.left.priority > node.priority:
                    node = self._rotate_right(node)
                    node.right = _delete(node.right, key)
                else:
                    node = self._rotate_left(node)
                    node.left = _delete(node.left, key)

            return node

        self.root = _delete(self.root, key)

    def save_diagram(self, filename: str, directory: str = OUTPUT_DIR):
        dot = graphviz.Digraph(comment='Treap')

        def _add_edges(node):
            if node:
                node_id = str(id(node))
                label = f"K: {node.key} | P: {node.priority}"
                dot.node(node_id, label=label, shape="record")

                if node.left:
                    left_id = str(id(node.left))
                    dot.edge(node_id, left_id, label="L")
                    _add_edges(node.left)
                if node.right:
                    right_id = str(id(node.right))
                    dot.edge(node_id, right_id, label="R")
                    _add_edges(node.right)

        if self.root:
            _add_edges(self.root)

        filepath = dot.render(filename=filename, directory=directory, format='png', cleanup=True)
        print(f"Diagrama Treap salvo em: {filepath}")
        return dot

In [4]:
treap = TreapInstrumentada()
for val in [10, 5, 20, 3, 7, 15, 25]:
    treap.insert(val)

dot_treap = treap.save_diagram("treap_teste", directory=OUTPUT_DIR_TESTE)
display(dot_treap)

NameError: name 'KDTreeInstrumentada' is not defined